# nanofable: ternary vs fp16 emergence frontier (Kaggle)
GPU **on**, Internet **on**, Persistence **Files only**. All logic lives in the repo (`nanofable.kaggle`); these cells just call it.

## 1. Sync repo + bootstrap (deps, artifact checks, data build)

In [ ]:
import os, subprocess, sys
REPO = "https://github.com/adit-rah/nanofable.git"; DIR = "/kaggle/working/nanofable"; BR = "main"
if os.path.isdir(os.path.join(DIR, ".git")):
    subprocess.run(["git","-C",DIR,"fetch","-q","--depth","1","origin",BR], check=True)
    subprocess.run(["git","-C",DIR,"reset","-q","--hard",f"origin/{BR}"], check=True)
else:
    subprocess.run(["rm","-rf",DIR]); subprocess.run(["git","clone","-q","--depth","1","-b",BR,REPO,DIR], check=True)
sys.path.insert(0, os.path.join(DIR, "src")); sys.path.insert(0, DIR); os.chdir(DIR)
print("commit", subprocess.run(["git","-C",DIR,"rev-parse","--short","HEAD"], capture_output=True, text=True).stdout.strip())

from nanofable.kaggle import bootstrap
ctx = bootstrap(DIR)   # installs bitsandbytes, verifies frozen artifacts, builds data memmaps

## 2. Calibrate the judge (once, before sweeping)
Confirm good refs clear ~4.0, garbage well below, and **intra-judge std < good−bad gap**. Set `use_33m=True` to add the TinyStories-33M reference.

In [ ]:
from nanofable.kaggle import calibrate
calibrate(ctx, use_33m=False)

## 3. Train the sweep (idempotent: re-run next session to resume)
On T4×2 this trains two runs at once (one worker per GPU; latest progress echoes as `[gpu0]`/`[gpu1]`, full logs in `runs/worker{i}.log`). Single-GPU accelerators fall back to sequential. `ONLY` limits a session; lower `micro_rows` if the large tier OOMs.

In [ ]:
from nanofable.sweep import run_sweep_parallel, sweep_matrix
print("matrix:", sweep_matrix())
ONLY = None   # e.g. [("tiny","fp16",0), ("tiny","ternary",0)] for a cheap first pass
run_sweep_parallel(ctx.runs_dir, ctx.train_path, ctx.val_path, only=ONLY,
                   total_tokens=500_000_000, tokens_per_step=65536, peak_lr=3e-4, micro_rows=16)

## 4. Judge every finished run (loads the judge once)

In [ ]:
from nanofable.kaggle import evaluate_all
evaluate_all(ctx)

## 5. Frontier plot + deliverables

In [ ]:
from nanofable.plotting import collect_results, plot_frontier
from nanofable.kaggle import save_deliverables
from IPython.display import Image
res = collect_results(ctx.runs_dir)
print("HEADLINE:", plot_frontier(res, "/kaggle/working/frontier.png"))
save_deliverables(ctx)
Image("/kaggle/working/frontier.png")